Objective: develop a non-trivial solution to a proposed topic
○ Develop all the objectives of your chosen project
○ Show critical thinking and problem solving skills
○ Introduce small novelty in the approach, feel free to experiment

This is a hands-on project, not a survey
○ Focus on methodology and approach



> Next steps: Implement DDPM (google model)

## Lightweight Diffusion Models <br><sub>Accelerating Training/Inference for Resource-Constrained Environments</sub>

### Imports

In [ ]:
import math
import os
import copy
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, utils
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torch.utils.data import DataLoader
from tqdm import tqdm

### Globals

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Pretrained google/ddpm-cifar10-32 weights (local HuggingFace cache).
# These are the only weights we depend on; everything else is plain PyTorch.
WEIGHTS_PATH = os.path.expanduser(
    "~/.cache/huggingface/hub/models--google--ddpm-cifar10-32"
    "/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80"
    "/diffusion_pytorch_model.bin"
)

# Diffusion schedule (linear beta, matching the original DDPM config exactly).
NUM_TRAIN_TIMESTEPS = 1000
BETA_START = 0.0001
BETA_END = 0.02

# Batch size for the baseline DDPM-vs-DDIM image comparison.
GEN_BATCH = 64

print(f"Device: {DEVICE}")

### Utils

In [ ]:
def linear_alphas_cumprod(beta_start=BETA_START, beta_end=BETA_END, num_steps=NUM_TRAIN_TIMESTEPS):
    """alpha_bar_t = cumprod(1 - beta_t) for a linear beta schedule."""
    betas = torch.linspace(beta_start, beta_end, num_steps, dtype=torch.float64)
    return torch.cumprod(1.0 - betas, dim=0).float()


def make_training_grid(n_steps, num_train=NUM_TRAIN_TIMESTEPS):
    """Descending DDIM timestep grid (noisy -> clean), matching DDIMScheduler.set_timesteps."""
    step = num_train // n_steps
    ts = (np.arange(0, n_steps) * step).round().astype(np.int64)[::-1].copy()
    return torch.from_numpy(ts)


def ddim_step(eps, x_t, a_s, a_e):
    """One deterministic DDIM step (eta=0): x_t -> x_prev. alphas broadcastable to (B,1,1,1)."""
    x0 = (x_t - (1 - a_s).sqrt() * eps) / a_s.sqrt()
    return a_e.sqrt() * x0 + (1 - a_e).sqrt() * eps


def save_grid(tensor, path, title):
    """Save and show a square grid of images that live in [-1, 1]."""
    imgs = ((tensor.clamp(-1, 1) + 1) / 2).permute(0, 2, 3, 1).cpu().numpy()
    n = int(math.sqrt(len(imgs)))
    fig, axes = plt.subplots(n, n, figsize=(n, n))
    for ax, img in zip(axes.flat, imgs):
        ax.imshow(img)
        ax.axis("off")
    fig.suptitle(title, fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  => saved {path}")

### Data

In [ ]:
def build_dataset(train=True):
    """CIFAR-10 normalized to [-1, 1] with horizontal-flip augmentation."""
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    return datasets.CIFAR10(root="./data", train=train, download=True, transform=transform)

### Network

In [ ]:
# --- Building blocks --------------------------------------------------------
# Timestep encoding + the primitive layers (ResNet, self-attention, sampling).

def sinusoidal_embedding(timesteps, dim, downscale_freq_shift=1.0):
    """Fixed sine/cosine timestep encoding (flip_sin_to_cos=False, freq_shift=1)."""
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000)
        * torch.arange(half, dtype=torch.float32, device=timesteps.device)
        / (half - downscale_freq_shift)
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimestepEmbedding(nn.Module):
    """Learned MLP that maps the fixed sinusoidal encoding into the model width."""
    def __init__(self, in_ch, embed_dim):
        super().__init__()
        self.linear_1 = nn.Linear(in_ch, embed_dim)
        self.act = nn.SiLU()
        self.linear_2 = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        return self.linear_2(self.act(self.linear_1(x)))


class ResnetBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch, temb_ch, groups=32, eps=1e-6):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch, eps=eps, affine=True)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_emb_proj = nn.Linear(temb_ch, out_ch)
        self.norm2 = nn.GroupNorm(groups, out_ch, eps=eps, affine=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.nonlinearity = nn.SiLU()
        self.conv_shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x, temb):
        h = self.conv1(self.nonlinearity(self.norm1(x)))
        h = h + self.time_emb_proj(self.nonlinearity(temb))[:, :, None, None]
        h = self.conv2(self.nonlinearity(self.norm2(h)))
        if self.conv_shortcut is not None:
            x = self.conv_shortcut(x)
        return x + h


class AttentionBlock(nn.Module):
    """Single-head spatial self-attention. Key names: group_norm/query/key/value/proj_attn."""
    def __init__(self, ch, groups=32, eps=1e-6):
        super().__init__()
        self.group_norm = nn.GroupNorm(groups, ch, eps=eps, affine=True)
        self.query = nn.Linear(ch, ch)
        self.key = nn.Linear(ch, ch)
        self.value = nn.Linear(ch, ch)
        self.proj_attn = nn.Linear(ch, ch)
        self.scale = ch ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.group_norm(x).view(B, C, -1).transpose(1, 2)   # (B, N, C)
        q, k, v = self.query(h), self.key(h), self.value(h)
        attn = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * self.scale, dim=-1)
        h = self.proj_attn(torch.bmm(attn, v))
        return x + h.transpose(1, 2).view(B, C, H, W)


class Downsample2D(nn.Module):
    """Stride-2 conv with asymmetric padding (downsample_padding=0 in the config)."""
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=0)

    def forward(self, x):
        return self.conv(F.pad(x, (0, 1, 0, 1)))


class Upsample2D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2.0, mode="nearest"))

In [ ]:
# --- Encoder / decoder blocks ----------------------------------------------
# Down/up blocks push and pop skip connections onto a shared stack. The "Attn"
# variants interleave a self-attention block after each ResNet.

class DownBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch, temb_ch, num_layers=2, add_downsample=True):
        super().__init__()
        self.resnets = nn.ModuleList([
            ResnetBlock2D(in_ch if i == 0 else out_ch, out_ch, temb_ch)
            for i in range(num_layers)
        ])
        self.downsamplers = nn.ModuleList([Downsample2D(out_ch)]) if add_downsample else None

    def forward(self, x, temb):
        skips = ()
        for r in self.resnets:
            x = r(x, temb)
            skips += (x,)
        if self.downsamplers:
            for d in self.downsamplers:
                x = d(x)
            skips += (x,)
        return x, skips


class AttnDownBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch, temb_ch, num_layers=2, add_downsample=True):
        super().__init__()
        self.resnets = nn.ModuleList([
            ResnetBlock2D(in_ch if i == 0 else out_ch, out_ch, temb_ch)
            for i in range(num_layers)
        ])
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in range(num_layers)])
        self.downsamplers = nn.ModuleList([Downsample2D(out_ch)]) if add_downsample else None

    def forward(self, x, temb):
        skips = ()
        for r, a in zip(self.resnets, self.attentions):
            x = a(r(x, temb))
            skips += (x,)
        if self.downsamplers:
            for d in self.downsamplers:
                x = d(x)
            skips += (x,)
        return x, skips


class UNetMidBlock2D(nn.Module):
    def __init__(self, ch, temb_ch):
        super().__init__()
        self.resnets = nn.ModuleList([ResnetBlock2D(ch, ch, temb_ch),
                                      ResnetBlock2D(ch, ch, temb_ch)])
        self.attentions = nn.ModuleList([AttentionBlock(ch)])

    def forward(self, x, temb):
        x = self.attentions[0](self.resnets[0](x, temb))
        return self.resnets[1](x, temb)


class UpBlock2D(nn.Module):
    def __init__(self, resnet_specs, temb_ch, add_upsample=True):
        super().__init__()
        self.resnets = nn.ModuleList([ResnetBlock2D(ic, oc, temb_ch) for ic, oc in resnet_specs])
        self.upsamplers = nn.ModuleList([Upsample2D(resnet_specs[-1][1])]) if add_upsample else None

    def forward(self, x, temb, skip_stack):
        for r in self.resnets:
            x = r(torch.cat([x, skip_stack.pop()], dim=1), temb)
        if self.upsamplers:
            for u in self.upsamplers:
                x = u(x)
        return x


class AttnUpBlock2D(nn.Module):
    def __init__(self, resnet_specs, temb_ch, add_upsample=True):
        super().__init__()
        out_ch = resnet_specs[0][1]
        self.resnets = nn.ModuleList([ResnetBlock2D(ic, oc, temb_ch) for ic, oc in resnet_specs])
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in resnet_specs])
        self.upsamplers = nn.ModuleList([Upsample2D(out_ch)]) if add_upsample else None

    def forward(self, x, temb, skip_stack):
        for r, a in zip(self.resnets, self.attentions):
            x = a(r(torch.cat([x, skip_stack.pop()], dim=1), temb))
        if self.upsamplers:
            for u in self.upsamplers:
                x = u(x)
        return x

In [ ]:
class UNet2DModel(nn.Module):
    """
    Pure-PyTorch UNet matching google/ddpm-cifar10-32. The submodule and parameter
    names line up with the diffusers checkpoint, so the pretrained state_dict loads
    with strict=True.

    block_out_channels=[128, 256, 256, 256], layers_per_block=2
    down: [DownBlock2D, AttnDownBlock2D, DownBlock2D, DownBlock2D]
    up  : [UpBlock2D,   UpBlock2D,       AttnUpBlock2D, UpBlock2D]

    forward(x, t) -> predicted noise (epsilon), shape (B, 3, 32, 32).
    """
    def __init__(self):
        super().__init__()
        temb_dim = 512
        self.conv_in = nn.Conv2d(3, 128, 3, padding=1)
        self.time_embedding = TimestepEmbedding(128, temb_dim)

        self.down_blocks = nn.ModuleList([
            DownBlock2D(128, 128, temb_dim, add_downsample=True),
            AttnDownBlock2D(128, 256, temb_dim, add_downsample=True),
            DownBlock2D(256, 256, temb_dim, add_downsample=True),
            DownBlock2D(256, 256, temb_dim, add_downsample=False),
        ])
        self.mid_block = UNetMidBlock2D(256, temb_dim)

        # in_ch in each (in, out) spec already includes the concatenated skip
        # (e.g. 512 = 256 upsampled + 256 skip; 384 = 256 + 128).
        self.up_blocks = nn.ModuleList([
            UpBlock2D([(512, 256), (512, 256), (512, 256)], temb_dim, add_upsample=True),
            UpBlock2D([(512, 256), (512, 256), (512, 256)], temb_dim, add_upsample=True),
            AttnUpBlock2D([(512, 256), (512, 256), (384, 256)], temb_dim, add_upsample=True),
            UpBlock2D([(384, 128), (256, 128), (256, 128)], temb_dim, add_upsample=False),
        ])

        self.conv_norm_out = nn.GroupNorm(32, 128, eps=1e-6, affine=True)
        self.conv_act = nn.SiLU()
        self.conv_out = nn.Conv2d(128, 3, 3, padding=1)

    def forward(self, x, t):
        temb = self.time_embedding(sinusoidal_embedding(t, 128))
        x = self.conv_in(x)

        skip_stack = [x]
        for block in self.down_blocks:
            x, skips = block(x, temb)
            skip_stack.extend(skips)

        x = self.mid_block(x, temb)

        for block in self.up_blocks:
            x = block(x, temb, skip_stack)

        return self.conv_out(self.conv_act(self.conv_norm_out(x)))


def load_unet(weights_path=WEIGHTS_PATH, device=DEVICE):
    """Build the UNet and load weights (defaults to the pretrained google checkpoint)."""
    model = UNet2DModel().to(device)
    state = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=True)
    return model

### Train

In [ ]:
def progressive_distill(teacher, *, grid_steps, teacher_jump, eval_steps,
                        loss_type="mse_position", lr=1e-5, epochs=5,
                        grad_clip=None, batch_size=32, save_path=None):
    """
    One stage of progressive distillation.

    The teacher takes `teacher_jump` DDIM steps on a `grid_steps`-point grid; the
    student (a clone of the teacher) learns to reproduce that move in a single step,
    which halves/thirds the number of inference steps needed.

    loss_type:
      "mse_position" -> MSE between the student's 1-step result and the teacher's
                        multi-step result            (stages 1000->25 and 25->12)
      "huber_x0"     -> Huber loss on the implied x0 predictions, with gradient
                        clipping                       (stage 12->8)
    """
    ac = linear_alphas_cumprod().to(DEVICE)
    timesteps = make_training_grid(grid_steps).to(DEVICE)

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    student = copy.deepcopy(teacher).to(DEVICE)
    student.train()
    for p in student.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(student.parameters(), lr=lr)
    loader = DataLoader(build_dataset(), batch_size=batch_size, shuffle=True, drop_last=True)

    print(f"Distilling: {teacher_jump}-step teacher jump on a {grid_steps}-step grid "
          f"-> {eval_steps}-step student  (loss={loss_type})")

    for epoch in range(epochs):
        pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}")
        for imgs, _ in pbar:
            imgs = imgs.to(DEVICE)
            B = imgs.shape[0]
            noise = torch.randn_like(imgs)

            # Random start, leaving room for `teacher_jump` steps ahead on the grid.
            idx = torch.randint(0, len(timesteps) - teacher_jump, (B,), device=DEVICE)
            t_start = timesteps[idx]
            t_target = timesteps[idx + teacher_jump]
            a0 = ac[t_start].view(-1, 1, 1, 1)
            a_target = ac[t_target].view(-1, 1, 1, 1)

            x_t = a0.sqrt() * imgs + (1 - a0).sqrt() * noise

            # Teacher: `teacher_jump` sequential DDIM steps (frozen, no grad).
            with torch.no_grad():
                x_ref = x_t.clone()
                for i in range(teacher_jump):
                    t_curr = timesteps[idx + i]
                    t_next = timesteps[idx + i + 1]
                    eps = teacher(x_ref, t_curr)
                    a_s = ac[t_curr].view(-1, 1, 1, 1)
                    a_e = ac[t_next].view(-1, 1, 1, 1)
                    x_ref = ddim_step(eps, x_ref, a_s, a_e)

            # Student: a single jump from t_start to t_target.
            s_eps = student(x_t, t_start)
            x0_student = (x_t - (1 - a0).sqrt() * s_eps) / a0.sqrt()

            if loss_type == "mse_position":
                x_fast = a_target.sqrt() * x0_student + (1 - a_target).sqrt() * s_eps
                loss = F.mse_loss(x_fast, x_ref.detach())
            elif loss_type == "huber_x0":
                # x0 the teacher implies, expressed through the student's noise direction.
                x0_target = (x_ref - (1 - a_target).sqrt() * s_eps.detach()) / a_target.sqrt()
                loss = F.huber_loss(x0_student, x0_target.detach(), delta=1.0)
            else:
                raise ValueError(f"unknown loss_type: {loss_type}")

            optimizer.zero_grad()
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(student.parameters(), grad_clip)
            optimizer.step()
            pbar.set_postfix({"loss": f"{loss.item():.7f}"})

    if save_path:
        torch.save(student.state_dict(), save_path)
        print(f"Saved {save_path}")

    return student

Progressive distillation runs in three sequential stages, each halving (or
thirding) the number of inference steps. Every stage loads the previous stage's
checkpoint as its frozen teacher, so run them in order.

| Stage | Steps | Teacher | Grid | Jump | Loss |
|-------|-------|---------|------|------|------|
| 1 | 1000 → 25 | google DDPM | 100 | 4 | MSE on position |
| 2 | 25 → 12 | `fast_professor_21_final.pt` | 24 | 2 | MSE on position |
| 3 | 12 → 8 | `fast_professor_12step.pt` | 24 | 3 | Huber on x0 + grad clip |

In [ ]:
# Stage 1: 1000 -> 25 steps  (teacher = original google DDPM, 4-step jump)
teacher = load_unet()  # pretrained google/ddpm-cifar10-32
student_25 = progressive_distill(
    teacher,
    grid_steps=100, teacher_jump=4, eval_steps=25,
    loss_type="mse_position", lr=1e-5, epochs=5,
    save_path="fast_professor_21_final.pt",
)

In [ ]:
# Stage 2: 25 -> 12 steps  (2-step teacher jump)
teacher_25 = load_unet("fast_professor_21_final.pt")
student_12 = progressive_distill(
    teacher_25,
    grid_steps=24, teacher_jump=2, eval_steps=12,
    loss_type="mse_position", lr=1e-5, epochs=5,
    save_path="fast_professor_12step.pt",
)

In [ ]:
# Stage 3: 12 -> 8 steps  (3-step teacher jump, Huber-on-x0 loss + grad clipping)
teacher_12 = load_unet("fast_professor_12step.pt")
student_8 = progressive_distill(
    teacher_12,
    grid_steps=24, teacher_jump=3, eval_steps=8,
    loss_type="huber_x0", lr=8e-6, epochs=5, grad_clip=1.0,
    save_path="fast_professor_8step.pt",
)

### Evaluation

In [ ]:
# --- Full-trajectory schedulers (used only for the original-model baseline) ---

class DDPMSchedulerFull:
    """Stochastic DDPM reverse process over all 1000 steps (variance_type=fixed_large)."""
    def __init__(self):
        self.alphas_cumprod = linear_alphas_cumprod()
        self.timesteps = torch.arange(NUM_TRAIN_TIMESTEPS - 1, -1, -1)

    @torch.no_grad()
    def step(self, eps, t, x_t):
        a_t = self.alphas_cumprod[t].to(x_t.device)
        a_prev = self.alphas_cumprod[t - 1].to(x_t.device) if t > 0 else torch.tensor(1.0)
        beta_t = 1 - a_t / a_prev
        x0 = ((x_t - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
        coef1 = a_prev.sqrt() * beta_t / (1 - a_t)
        coef2 = (a_t / a_prev).sqrt() * (1 - a_prev) / (1 - a_t)
        mean = coef1 * x0 + coef2 * x_t
        if t > 0:
            mean = mean + beta_t.sqrt() * torch.randn_like(x_t)
        return mean


class DDIMSchedulerFull:
    """Deterministic DDIM (eta=0) over a strided grid, same linear betas."""
    def __init__(self, num_inference_steps=30):
        self.alphas_cumprod = linear_alphas_cumprod()
        self._step = NUM_TRAIN_TIMESTEPS // num_inference_steps
        ts = (np.arange(0, num_inference_steps) * self._step).round().astype(np.int64)[::-1].copy()
        self.timesteps = torch.from_numpy(ts)

    @torch.no_grad()
    def step(self, eps, t, x_t):
        a_t = self.alphas_cumprod[t].to(x_t.device)
        prev_t = t - self._step
        a_prev = self.alphas_cumprod[prev_t].to(x_t.device) if prev_t >= 0 else torch.tensor(1.0)
        x0 = (x_t - (1 - a_t).sqrt() * eps) / a_t.sqrt()
        return a_prev.sqrt() * x0 + (1 - a_prev).sqrt() * eps


@torch.no_grad()
def run_sampler(model, scheduler, batch, device, seed=SEED):
    """Run a full reverse-diffusion loop with the given scheduler. Returns (images, seconds)."""
    torch.manual_seed(seed)
    x = torch.randn(batch, 3, 32, 32, device=device)
    t0 = time.time()
    for t in scheduler.timesteps:
        t_b = torch.full((batch,), t.item(), device=device, dtype=torch.long)
        x = scheduler.step(model(x, t_b), t.item(), x)
    return x, time.time() - t0

In [ ]:
@torch.no_grad()
def generate_n_steps(model, alphas_cumprod, batch_size, device, n_steps):
    """Generate a batch with n_steps linearly spaced DDIM steps (the few-step sampler)."""
    model.eval()
    x = torch.randn(batch_size, 3, 32, 32, device=device)
    timesteps = torch.linspace(NUM_TRAIN_TIMESTEPS - 1, 0, n_steps, dtype=torch.long, device=device)
    alphas = alphas_cumprod.to(device)
    for i, t in enumerate(timesteps):
        t_val = t.item()
        t_b = torch.full((batch_size,), t_val, device=device, dtype=torch.long)
        eps = model(x, t_b)
        a_s = alphas[t_val].view(1, 1, 1, 1)
        if i == len(timesteps) - 1:
            x = (x - (1 - a_s).sqrt() * eps) / a_s.sqrt()
        else:
            a_e = alphas[timesteps[i + 1].item()].view(1, 1, 1, 1)
            x = ddim_step(eps, x, a_s, a_e)
    return x.clamp(-1, 1)


def run_eval(model, dataset, device, num_samples=10000, steps=25, gen_batch=64):
    """FID + IS for `model` generated at `steps` DDIM steps vs. real CIFAR-10."""
    print(f"\n--- Evaluating {num_samples} samples at {steps} steps ---")
    ac = linear_alphas_cumprod()
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    is_metric = InceptionScore(normalize=True).to(device)

    # Real images (streamed in [0, 1]).
    real_loader = DataLoader(dataset, batch_size=128, shuffle=False)
    real_count = 0
    with torch.no_grad():
        for imgs, _ in tqdm(real_loader, desc="Real images", leave=False):
            if real_count >= num_samples:
                break
            batch = (imgs[:num_samples - real_count].to(device) + 1.0) / 2.0
            fid.update(batch, real=True)
            real_count += batch.shape[0]

    # Fake images (generated in chunks).
    fake_count = 0
    with torch.no_grad():
        while fake_count < num_samples:
            bs = min(gen_batch, num_samples - fake_count)
            samples = (generate_n_steps(model, ac, bs, device, steps) + 1.0) / 2.0
            fid.update(samples, real=False)
            is_metric.update(samples)
            fake_count += bs
            print(f"  generated {fake_count}/{num_samples}", end="\r")

    print(f"\nFID: {fid.compute().item():.4f} | IS: {is_metric.compute()[0].item():.4f}")

#### Baseline: DDPM (1000 steps) vs DDIM (30 steps)

Same pretrained weights, two samplers. DDIM swaps in for free — no retraining — and
gives a large speedup with only a small quality drop.

In [ ]:
# Baseline: original google/ddpm-cifar10-32 — DDPM 1000 steps vs DDIM 30 steps.
model = load_unet()
model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

imgs_ddpm, t_ddpm = run_sampler(model, DDPMSchedulerFull(), GEN_BATCH, DEVICE)
print(f"DDPM 1000 steps: {t_ddpm:.2f}s")
save_grid(imgs_ddpm, "ddpm_1000_steps.png", f"DDPM 1000 steps ({t_ddpm:.2f}s)")

imgs_ddim, t_ddim = run_sampler(model, DDIMSchedulerFull(30), GEN_BATCH, DEVICE)
print(f"DDIM 30 steps:   {t_ddim:.2f}s")
save_grid(imgs_ddim, "ddim_30_steps.png", f"DDIM 30 steps ({t_ddim:.2f}s)")

print(f"Speedup: {t_ddpm / t_ddim:.1f}x")

#### Distilled student (FID / IS)

Quantitative scores for a distilled checkpoint, using the few-step DDIM sampler.

In [ ]:
# Evaluate a distilled student (FID / IS over 10k samples).
# Point this at whichever checkpoint you want to score.
student = load_unet("fast_professor_8step.pt")
run_eval(student, build_dataset(), DEVICE, num_samples=10000, steps=8)